# Traffic Regulation Order (TRO) Extraction Pipeline (Part 2: CSV to Geocoded Segments)

Part 1 turned a scanned TRO PDF into a structured CSV, one row per restriction, each
describing a street segment in plain English (e.g. *"From the extended line of the northeast
kerb of Stirling Road eastwards for a distance of 58 metres or thereby"*). This notebook turns
those descriptions into real line geometries you can map, using **OS Open Roads** - Ordnance
Survey's free, official UK road network dataset, as the reference geometry.

## The approach

1. **Parse** each restriction's text into a structured reference: which street(s) it's
   measured from, which compass direction, and how far.
2. **Locate the junction** where the restricted street meets its reference street, using OS
   Open Roads' topology (RoadLink features share a RoadNode wherever two streets cross).
3. **Walk along the street** from that junction, either a fixed distance in a compass
   direction, or directly between two named reference junctions, accumulating across
   multiple linked road segments as needed, since OS Open Roads splits long streets into many
   short `RoadLink` features.
4. **Export** the result as GeoJSON and an interactive Folium map.

## Getting OS Open Roads
1. Go to the [OS Data Hub — OS Open Roads download page](https://osdatahub.os.uk/downloads/open/OpenRoads) and create a free account if you don't have one.
2. Choose **GeoPackage** format.
3. Use the map picker to select the area covering your TRO (Airdrie / North Lanarkshire falls
   within the **NS** 100km National Grid square).
4. Save the downloaded `.gpkg` file to `../Geodata/os_open_roads_north_lanarkshire.gpkg`
   (or update `ROADS_GPKG_PATH` below).

The file contains two layers we'll use: `road_link` (street centerline geometries, with a
`name1` field) and `road_node` (junction points).


## 0. Setup

In [ ]:
# pip install geopandas shapely networkx folium pandas pyproj
import math
import re
from difflib import get_close_matches
from pathlib import Path

import folium
import geopandas as gpd
import networkx as nx
import pandas as pd
from shapely.geometry import LineString, Point


In [ ]:
# --- Config: edit these for your files ---
ORDER_ID = "NL001-airdrie-2018"
PROCESSED_CSV_PATH = f"../Final/{ORDER_ID}_processed_all_fields.csv"
ROADS_LINKS_GPKG_PATH = "../Geodata/os_open_roads_north_lanarkshire.gpkg"
ROADS_NODES_GPKG_PATH = "../Geodata/os_open_roads_node_north_lanarkshire.gpkg"

OUTPUT_GEOJSON = f"../Final/{ORDER_ID}_segments.geojson"
OUTPUT_HTML_MAP = f"../Final/{ORDER_ID}_map.html"

FUZZY_MATCH_CUTOFF = 0.8
MAX_BEARING_DEVIATION = 60

COMPASS_BEARINGS = {
    "north": 0, "northeast": 45, "east": 90, "southeast": 135,
    "south": 180, "southwest": 225, "west": 270, "northwest": 315,
}

## 1. Load the structured bylaw CSV from Part 1


In [ ]:
df = pd.read_csv(PROCESSED_CSV_PATH)
print(df.shape)
df[["Area", "Street", "Side_of_Street", "Length"]].head(10)


## 2. Load OS Open Roads and match street names

Street names in the TRO text won't always match the OS `name1` field exactly — most commonly
because the TRO includes a route number prefix (e.g. `"A73 Motherwell Street"`) that OS Open
Roads doesn't. We normalize both sides and fall back to fuzzy matching for anything that
doesn't match exactly.


In [ ]:
road_link = gpd.read_file(ROADS_LINKS_GPKG_PATH, layer="os_open_roads_north_lanarkshire")
road_node = gpd.read_file(ROADS_NODES_GPKG_PATH, layer="oproad_gb__road_node")

print(f"CRS: {road_link.crs}")
print(f"{len(road_link)} road links, {len(road_node)} road nodes")
road_link[["id", "name_1", "start_node", "end_node", "length"]].head()

In [ ]:
def normalize_street_name(name):
    name = str(name).strip()
    name = re.sub(r"^(?:the\s+)?[A-Z]\d+\s+", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\s+", " ", name)
    return name.strip().upper()


road_link["name1_norm"] = road_link["name_1"].map(normalize_street_name)
known_names = set(road_link["name1_norm"].dropna())

unmatched = []
for street in sorted(df["Street"].dropna().unique()):
    norm = normalize_street_name(street)
    if norm in known_names:
        continue
    close = get_close_matches(norm, known_names, n=1, cutoff=FUZZY_MATCH_CUTOFF)
    if not close:
        unmatched.append(street)

print(f"{df['Street'].nunique()} unique streets in the CSV, {len(unmatched)} with no OS Open Roads match")
unmatched

## 3. Parse each restriction's reference geometry

Two sentence patterns show up in the data:

- **One reference + direction**: *"From the extended line of the northeast kerb of Stirling
  Road eastwards for a distance of 58 metres or thereby."* → walk 58m east from the
  Stirling Road junction.
- **Two references, explicit span**: *"From the extended line of the west kerb of Wellwynd
  eastwards to the extended line of the west kerb of Bank Street, a distance of 111 metres."*
  → trace directly between the two junctions (more reliable — no direction math needed).

Anything neither pattern matches is flagged rather than silently skipped, so it can be
reviewed and handled manually or with a targeted regex/LLM fallback later.


In [ ]:
TWO_REF = re.compile(
    r"from the extended line of the .*? kerb of (?:the )?(?P<ref1>.+?),?\s+"
    r"(?P<dir1>north|south|east|west|north[\s-]?east|north[\s-]?west|south[\s-]?east|south[\s-]?west)\s?-?\s?wards?\s+"
    r"to the extended line of the .*? kerb (?:of )?(?P<ref2>.+?),?\s+"
    r"a distance of (?P<dist>\d+)",
    re.IGNORECASE,
)

ONE_REF_PATTERNS = [
    re.compile(
        r"from the extended line of the .*? kerb of (?:the )?(?P<ref1>.+?),?\s+"
        r"(?P<dir1>north|south|east|west|north[\s-]?east|north[\s-]?west|south[\s-]?east|south[\s-]?west)\s?-?\s?wards?\s+"
        r"for a distance of (?P<dist>\d+)",
        re.IGNORECASE,
    ),
    re.compile(
        r"from its junction with (?:the )?(?P<ref1>.+?),?\s+"
        r"(?P<dir1>north|south|east|west|north[\s-]?east|north[\s-]?west|south[\s-]?east|south[\s-]?west)\s?-?\s?wards?\s+"
        r"for a distance of (?P<dist>\d+)",
        re.IGNORECASE,
    ),
]


def normalize_direction(raw_dir):
    return raw_dir.lower().replace(" ", "").replace("-", "")


parsed_rows = []
for _, row in df.iterrows():
    text = str(row["Raw_Text"]).strip()
    m2 = TWO_REF.search(text)
    m1 = None
    if not m2:
        for pattern in ONE_REF_PATTERNS:
            m1 = pattern.search(text)
            if m1:
                break

    if m2:
        parsed_rows.append({
            "ref1": m2.group("ref1").strip(),
            "direction": normalize_direction(m2.group("dir1")),
            "ref2": m2.group("ref2").strip(),
            "distance_m": float(m2.group("dist")),
            "pattern": "two_ref",
        })
    elif m1:
        parsed_rows.append({
            "ref1": m1.group("ref1").strip(),
            "direction": normalize_direction(m1.group("dir1")),
            "ref2": None,
            "distance_m": float(m1.group("dist")),
            "pattern": "one_ref",
        })
    else:
        parsed_rows.append({"ref1": None, "direction": None, "ref2": None, "distance_m": None, "pattern": "unparsed"})

parsed_df = pd.DataFrame(parsed_rows)
df = pd.concat([df.reset_index(drop=True), parsed_df], axis=1)

print(df["pattern"].value_counts())
df.loc[df["pattern"] == "unparsed", ["Street", "Raw_Text"]]


## 4. Build the street-network helpers

Two small, reused helpers do the actual geometric work:

- `find_junction_node` — finds the shared `RoadNode` where two named streets meet.
- `build_street_graph` / `walk_from_node` / `trace_between_nodes` — build a small graph of
  just the `RoadLink` segments for one street name, then either walk a fixed distance in a
  compass direction, or trace the shortest path between two known nodes.


In [ ]:
def find_junction_node(street_a_norm, street_b_norm, road_link):
    links_a = road_link[road_link["name1_norm"] == street_a_norm]
    links_b = road_link[road_link["name1_norm"] == street_b_norm]
    nodes_a = set(links_a["start_node"]) | set(links_a["end_node"])
    nodes_b = set(links_b["start_node"]) | set(links_b["end_node"])
    shared = nodes_a & nodes_b
    return next(iter(shared)) if shared else None


def resolve_street_name(name, known_names):
    norm = normalize_street_name(name)
    if norm in known_names:
        return norm
    close = get_close_matches(norm, known_names, n=1, cutoff=FUZZY_MATCH_CUTOFF)
    return close[0] if close else None


node_points = dict(zip(road_node["id"], road_node.geometry))


def bearing_deg(p1, p2):
    dx, dy = p2.x - p1.x, p2.y - p1.y
    return math.degrees(math.atan2(dx, dy)) % 360


def angle_diff(a, b):
    d = abs(a - b) % 360
    return min(d, 360 - d)


_street_graph_cache = {}

def build_street_graph(street_name_norm):
    if street_name_norm in _street_graph_cache:
        return _street_graph_cache[street_name_norm]
    subset = road_link[road_link["name1_norm"] == street_name_norm]
    g = nx.Graph()
    for _, row in subset.iterrows():
        g.add_edge(row["start_node"], row["end_node"], geometry=row.geometry, length=row["length"])
    _street_graph_cache[street_name_norm] = g
    return g


def walk_from_node(start_node_id, street_name_norm, distance_m, compass_direction):
    g = build_street_graph(street_name_norm)
    if start_node_id not in g:
        return None
    target_bearing = COMPASS_BEARINGS[compass_direction]
    remaining = distance_m
    current_node = start_node_id
    visited_edges = set()
    path_coords = [node_points[current_node]]

    while remaining > 1e-6:
        candidates = []
        for neighbor in g.neighbors(current_node):
            edge_key = tuple(sorted((current_node, neighbor)))
            if edge_key in visited_edges:
                continue
            b = bearing_deg(node_points[current_node], node_points[neighbor])
            candidates.append((angle_diff(b, target_bearing), neighbor))
        if not candidates:
            break
        candidates.sort(key=lambda c: c[0])
        best_diff, next_node = candidates[0]
        if best_diff > MAX_BEARING_DEVIATION:
            break

        edge = g.edges[current_node, next_node]
        edge_len = edge["length"]
        visited_edges.add(tuple(sorted((current_node, next_node))))

        if edge_len <= remaining:
            path_coords.append(node_points[next_node])
            remaining -= edge_len
            current_node = next_node
        else:
            geom = edge["geometry"]
            start_pt = node_points[current_node]
            if Point(geom.coords[0]).distance(start_pt) > 1e-6:
                geom = LineString(list(geom.coords)[::-1])
            path_coords.append(geom.interpolate(remaining))
            remaining = 0

    if len(path_coords) < 2:
        return None
    return LineString(path_coords)


def trace_between_nodes(street_name_norm, node_a, node_b):
    g = build_street_graph(street_name_norm)
    if node_a not in g or node_b not in g:
        return None
    try:
        path = nx.shortest_path(g, node_a, node_b, weight="length")
    except nx.NetworkXNoPath:
        return None
    return LineString([node_points[n] for n in path])


## 5. Build a geometry for every parsed row

For each row: resolve the target street and reference street name(s) against OS Open Roads,
find the junction(s), then either walk the stated distance in the stated direction (one-ref
rows) or trace directly between two junctions (two-ref rows).


In [ ]:
results = []

for idx, row in df.iterrows():
    if row["pattern"] == "unparsed":
        results.append(None)
        continue

    target_norm = resolve_street_name(row["Street"], known_names)
    ref1_norm = resolve_street_name(row["ref1"], known_names)

    if target_norm is None or ref1_norm is None:
        results.append(None)
        continue

    junction1 = find_junction_node(target_norm, ref1_norm, road_link)
    if junction1 is None:
        results.append(None)
        continue

    if row["pattern"] == "two_ref":
        ref2_norm = resolve_street_name(row["ref2"], known_names)
        if ref2_norm is None:
            results.append(None)
            continue
        junction2 = find_junction_node(target_norm, ref2_norm, road_link)
        geometry = trace_between_nodes(target_norm, junction1, junction2) if junction2 else None
    else:
        geometry = walk_from_node(junction1, target_norm, row["distance_m"], row["direction"])

    results.append(geometry)

df["geometry"] = results

success = df["geometry"].notna().sum()
print(f"Built geometry for {success} / {len(df)} rows ({success / len(df):.0%})")
df.loc[df["geometry"].isna(), ["Street", "Raw_Text", "pattern"]].head(10)


## 6. Save the GeoJSON output


In [ ]:
geo_df = gpd.GeoDataFrame(df[df["geometry"].notna()].copy(), geometry="geometry", crs=road_link.crs)
geo_df = geo_df.to_crs(epsg=4326)  # WGS84 for GeoJSON / Folium

keep_cols = ["Area", "Bylaw_Schedule", "Street", "Side_of_Street", "Length", "Raw_Text",
             "Raw_Bylaw_Text", "TRO Title", "Effective Date", "geometry"]
geo_df = geo_df[[c for c in keep_cols if c in geo_df.columns]]

Path(OUTPUT_GEOJSON).parent.mkdir(parents=True, exist_ok=True)
geo_df.to_file(OUTPUT_GEOJSON, driver="GeoJSON")

print(f"Saved {len(geo_df)} segments to {OUTPUT_GEOJSON}")
geo_df.head()


In [ ]:
# Quick static preview before building the interactive map
geo_df.plot(figsize=(8, 8), linewidth=2)


## 7. Build an interactive Folium map


In [ ]:
schedule_colors = {}
palette = ["#e6194B", "#3cb44b", "#4363d8", "#f58231", "#911eb4", "#46f0f0", "#f032e6",
           "#bfef45", "#fabed4", "#469990", "#dcbeff", "#9A6324", "#800000", "#808000"]

for i, schedule in enumerate(sorted(geo_df["Bylaw_Schedule"].dropna().unique())):
    schedule_colors[schedule] = palette[i % len(palette)]

center = [(geo_df.total_bounds[1] + geo_df.total_bounds[3]) / 2,
          (geo_df.total_bounds[0] + geo_df.total_bounds[2]) / 2]
m = folium.Map(location=center, zoom_start=15, tiles="cartodbpositron")

for _, row in geo_df.iterrows():
    color = schedule_colors.get(row["Bylaw_Schedule"], "#333333")
    popup_text = (
        f"<b>{row['Street']}</b> ({row['Side_of_Street']})<br>"
        f"Schedule {row['Bylaw_Schedule']}<br>"
        f"{row['Raw_Bylaw_Text']}<br>"
        f"<i>{row['Raw_Text']}</i>"
    )
    folium.GeoJson(
        row.geometry,
        style_function=lambda _, color=color: {"color": color, "weight": 4},
        tooltip=f"{row['Street']} - Schedule {row['Bylaw_Schedule']}",
        popup=folium.Popup(popup_text, max_width=300),
    ).add_to(m)

m.save(OUTPUT_HTML_MAP)
print(f"Map saved to {OUTPUT_HTML_MAP}")
m
